# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step exploration of the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The notebook follows the recommended template, making all references using entity `@id` and demonstrates key techniques for loading and processing Croissant-based datasets.

### Dataset Source
- [FAIR^2 Dataset Croissant Schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Dataset Description:', metadata.description)
print('Dataset Identifier:', metadata.identifier)
print('Dataset Version:', metadata.version)


## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# Croissant datasets typically have 'recordSet' attribute listing the record sets by @id

from pprint import pprint

record_set_ids = getattr(metadata, 'recordSet', [])

if not record_set_ids:
    print("No record sets listed in metadata. Inferring from records() function.")
    # Try showing available record sets from the schema
    # For mlcroissant >= 1.0.1, Dataset.record_sets() returns a dict of RecordSet objects
    record_sets_dict = dataset.record_sets()
    print("Available Record Sets (@id):")
    for rs_id in record_sets_dict:
        print("-", rs_id)
        rs_obj = record_sets_dict[rs_id]
        print("  name:", getattr(rs_obj, 'name', '(none)'))
        if hasattr(rs_obj, 'field'):
            print("  Fields (@id):")
            for f in getattr(rs_obj, 'field', []):
                if hasattr(f, '@id'):
                    print("    -", f['@id'])
                elif hasattr(f, 'id'):
                    print("    -", f.id)
        print()
else:
    print("Record Sets listed in metadata:")
    pprint(record_set_ids)


### Quick Preview of Records
Display some sample records from one of the record sets using its `@id`.

In [ ]:
# We'll select the first available record set for preview

record_sets_dict = dataset.record_sets()
# Use first record set @id detected
rs_ids = list(record_sets_dict.keys())
if rs_ids:
    preview_rs_id = rs_ids[0]
    print(f"Showing sample records from record set '@id': {preview_rs_id}")

    for i, x in enumerate(dataset.records(record_set=preview_rs_id)):
        print(x)
        if i > 4:
            break
else:
    print("No record sets found to preview.")


## 3. Data Extraction

Load data from **all record sets** into DataFrames for analysis.
- Use the record set and field `@id`s from the overview above.
- All references should be by `@id`.

In [ ]:
# Extract all available record sets to DataFrames
record_sets_dict = dataset.record_sets()
record_set_ids = list(record_sets_dict.keys())

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame with columns: {df.columns.tolist()}")
        print(f"First 3 records:")
        print(df.head(3))
    else:
        print(f"No records found for record set {rs_id}")
    print()

# Choose the primary record set for further analysis
primary_rs_id = record_set_ids[0] if record_set_ids else None
if primary_rs_id:
    print(f"Columns in primary record set '{primary_rs_id}':", dataframes[primary_rs_id].columns.tolist())
    dataframes[primary_rs_id].head()


## 4. Exploratory Data Analysis (EDA)

- Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping.
- Reference all fields and columns by their `@id`.

In [ ]:
# EDA on the primary record set
df = dataframes.get(primary_rs_id)
if df is not None:
    # Find candidate numeric columns to analyze using their @id
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print('Numeric fields (@id):', numeric_candidates)
    
    # Example: filter on the first numeric column by @id
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        threshold = df[numeric_field_id].quantile(0.9)  # Use 90th percentile as example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
        print(filtered_df.head())

        # Normalize this numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized field '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        categorical_candidates = [col for col in df.columns if df[col].dtype == 'object' and df[col].nunique() < 10]
        if categorical_candidates:
            group_field_id = categorical_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean '{numeric_field_id}' grouped by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print("No numeric fields detected.")
else:
    print("No DataFrame found for primary record set.")


## 5. Visualization
Visualize distributions or relationships between selected fields (`@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use the field IDs found above
if df is not None and numeric_candidates:
    numeric_id = numeric_candidates[0]

    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_id], bins=10, kde=True)
    plt.title(f"Distribution of '{numeric_id}'")
    plt.xlabel(numeric_id)
    plt.ylabel('Frequency')
    plt.show()

    # Visualize a relationship between two numeric fields if available
    if len(numeric_candidates) >= 2:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(
            x=df[numeric_candidates[0]],
            y=df[numeric_candidates[1]]
        )
        plt.title(f"Scatter plot: '{numeric_candidates[0]}' vs '{numeric_candidates[1]}'")
        plt.xlabel(numeric_candidates[0])
        plt.ylabel(numeric_candidates[1])
        plt.show()


## 6. Conclusion

In this notebook, we have loaded the FAIR^2 dataset schema, explored its record sets and fields (referenced by their `@id`), and demonstrated extraction, filtering, normalization, grouping, and visualization of tabular clinical cancer data. All references strictly used entity `@id` per Croissant schema conventions.

This workflow can be adapted to any Croissant dataset or FAIR^2 data package, including those with multiple record sets or complex field structures. The `mlcroissant` library provides robust methods for metadata discovery, record streaming, and transformation suitable for clinical, biomedical, or broader research use.
